# Dataset Exploration Notebook

Inspect and validate the 6.042J dataset produced by the extraction pipeline.

**Sections**
1. Setup
2. Pipeline overview *(live in-memory demo + BM25 justification)*
3. Chunks — content passage quality
4. Questions — extraction quality
5. BM25 alignment — retrieval quality
6. Dataset summary

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

COURSE_ID = "6.042J"
PROCESSED  = ROOT / f"data/processed/{COURSE_ID}"
ALIGNED    = ROOT / f"data/aligned/{COURSE_ID}"
DATASET    = ROOT / "data/dataset.jsonl"

print("Paths OK")

## 2. Pipeline overview

The dataset is built in **3 steps**, each implemented as a standalone module:

```
Step 1 — EXTRACT   data/pipeline/extract.py
  content PDFs ──pdfplumber──► Chunk objects  (passage retrieval candidates)
  assignment / exam PDFs ────► RawQuestion objects

Step 2 — ALIGN     data/pipeline/align.py
  chunks + questions ──BM25Okapi──► aligned records (question + best-match passage)

Step 3 — FLATTEN   data/pipeline/flatten.py
  aligned records ──► flat dataset.jsonl  (one line = one evaluation item)
```

The three code cells below run each step **in-memory on a small sample** so you can see exactly what each module produces — no files are written.

### Why BM25 for passage alignment?

BM25 (Best Match 25) is a **probabilistic term-weighting retrieval function** that ranks passages by their relevance to a query (here: the question text).

**Formal definition** — the BM25 score for passage $d$ given query $q$ is:

$$\text{BM25}(d,q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \left(1 - b + b \cdot \frac{|d|}{\overline{|d|}}\right)}$$

where $f(t,d)$ is term frequency, $\overline{|d|}$ is average document length, $k_1 = 1.5$, $b = 0.75$ are tuning constants.

**Is BM25 state of the art?**

| Retriever | Type | Typical recall@100 | When to use |
|---|---|---|---|
| **BM25** | Sparse / lexical | ~75–80 % | No GPU, interpretable, domain-agnostic |
| DPR (Karpukhin et al., 2020) | Dense / neural | ~78–85 % | GPU available, in-domain fine-tuning |
| Contriever (Izacard et al., 2022) | Dense / unsupervised | ~80–88 % | No fine-tuning needed, best zero-shot |
| BM25 + DPR hybrid | Sparse + dense | ~85–90 % | Best recall, higher complexity |

Dense retrievers outperform BM25 on recall when the question and passage use different vocabulary (paraphrase-heavy). For **our use-case** — aligning questions written against a specific textbook — question terms heavily overlap with passage terms, which is exactly where BM25 excels.

**Key references:**
- Robertson & Zaragoza (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in Information Retrieval, 3(4), 333–389.
- Rajpurkar et al. (2016). *SQuAD: 100,000+ Questions for Machine Comprehension of Text.* EMNLP 2016. — BM25 used as the retrieval baseline for passage-level QA.
- Karpukhin et al. (2020). *Dense Passage Retrieval for Open-Domain Question Answering.* EMNLP 2020. — Shows DPR outperforms BM25 on *open-domain* QA; for closed-domain alignment BM25 remains competitive.
- Ma et al. (2021). *A Replication Study of Dense Passage Retrieval.* arXiv 2104.05740. — BM25 is a strong, often underestimated baseline.

> **Bottom line**: BM25 is not the absolute state-of-the-art for open-domain retrieval, but it is the **standard, well-justified baseline** for closed-domain passage alignment in educational QA pipelines, with no GPU requirement and full interpretability (you can see exactly why a passage was retrieved from its BM25 score).
Upgrading to a dense retriever is a natural future improvement once the corpus grows.

In [ ]:
# ── Step 1: EXTRACT — run on one content PDF + one assignment PDF ──────────
import sys; sys.path.insert(0, str(ROOT))
from data.pipeline.extract import ContentChunker, QuestionExtractor

COURSE_DIR = ROOT / "data/raw" / COURSE_ID
content_pdfs = sorted((COURSE_DIR / "content").glob("*.pdf"))
assignment_pdfs = sorted((COURSE_DIR / "assignments").glob("*.pdf"))

# --- Content: chunk the first content PDF
chunker = ContentChunker(chunk_offset=0)
sample_chunks = chunker.extract(content_pdfs[0])
print(f"Chunked '{content_pdfs[0].name}'  →  {len(sample_chunks)} chunks")
print()
c = sample_chunks[5]  # inspect chunk #5
print(f"chunk_id   : {c.chunk_id}")
print(f"topic      : {c.topic}")
print(f"tokens     : {c.token_count}")
print(f"text[:300] : {c.text[:300]}")
print()

# --- Questions: extract from the first assignment PDF
extractor = QuestionExtractor(source_type='assignment')
sample_questions = extractor.extract(assignment_pdfs[0])
print(f"Extracted '{assignment_pdfs[0].name}'  →  {len(sample_questions)} questions")
print()
q = sample_questions[0]
print(f"q_id         : {q.q_id}")
print(f"source_type  : {q.source_type}")
print(f"source_label : {q.source_label}")
print(f"options      : {q.options}")
print(f"text[:300]   : {q.text[:300]}")

In [ ]:
# ── Step 2: ALIGN — BM25 retrieval on the sample (no files written) ────────
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# Use the chunks from Step 1 as the corpus
corpus = [tokenize(c.text) for c in sample_chunks]
bm25 = BM25Okapi(corpus)

# Align the first 3 sample questions
for q in sample_questions[:3]:
    query = tokenize(q.text)
    scores = bm25.get_scores(query)
    best_idx = int(scores.argmax())
    best_score = float(scores[best_idx])
    best_chunk = sample_chunks[best_idx]
    low_conf = best_score < 5.0
    print(f"Question : {q.text[:100]}")
    print(f"  → chunk : {best_chunk.chunk_id} | topic: {best_chunk.topic}")
    print(f"  → BM25  : {best_score:.2f}{'  [LOW CONFIDENCE]' if low_conf else ''}")
    print(f"  → text  : {best_chunk.text[:150]}")
    print()

In [ ]:
# ── Step 3: FLATTEN — build the final record shape (in-memory) ─────────────
import json as _json

def make_flat_record(q, chunk, bm25_score, course_id):
    return {
        "question": {
            "id": q.q_id,
            "text": q.text,
            "options": q.options,
            "correct_answer": None,
            "metadata": {
                "course_id": course_id,
                "source_type": q.source_type,
                "source_label": q.source_label,
            },
        },
        "context": {
            "course_content": chunk.text,
            "metadata": {
                "chunk_id": chunk.chunk_id,
                "topic": chunk.topic,
                "bm25_score": bm25_score,
                "low_confidence": bm25_score < 5.0,
            },
        },
    }

# Build one record to show the final shape
q0 = sample_questions[0]
query = tokenize(q0.text)
scores = bm25.get_scores(query)
best_idx = int(scores.argmax())
record = make_flat_record(q0, sample_chunks[best_idx], float(scores[best_idx]), COURSE_ID)

print("Final record structure (one evaluation item):")
print(_json.dumps({
    k: v if k != 'context' else {
        'course_content': v['course_content'][:120] + '…',
        'metadata': v['metadata']
    }
    for k, v in record.items()
}, indent=2))

## 3. Chunks — content passage quality

In [ ]:
chunks = [json.loads(l) for l in (PROCESSED / "chunks.jsonl").open()]
df_chunks = pd.DataFrame(chunks)
print(f"Total chunks: {len(df_chunks)}")
df_chunks[["chunk_id", "source_file", "topic", "token_count"]].head(10)

In [ ]:
# Token count distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_chunks["token_count"], bins=40, color="#55A868", edgecolor="white")
ax.axvline(df_chunks["token_count"].median(), color="black", linestyle="--",
           label=f'median = {df_chunks["token_count"].median():.0f}')
ax.set_xlabel("Token count per chunk")
ax.set_ylabel("Count")
ax.set_title(f"Chunk size distribution — {COURSE_ID}")
ax.legend()
plt.tight_layout()
plt.show()
df_chunks["token_count"].describe()

In [ ]:
# Inspect a specific chunk — change IDX
IDX = 10
c = chunks[IDX]
print(f"chunk_id   : {c['chunk_id']}")
print(f"source_file: {c['source_file']}")
print(f"topic      : {c['topic']}")
print(f"tokens     : {c['token_count']}")
print()
print(c["text"])

## 4. Questions — extraction quality

In [ ]:
questions = [json.loads(l) for l in (PROCESSED / "questions_raw.jsonl").open()]
df_q = pd.DataFrame(questions)
print(f"Total questions: {len(df_q)}")
print(df_q["source_type"].value_counts().to_string())
print()
df_q[["q_id", "source_type", "source_label", "text"]].head(10)

In [ ]:
# Questions per source file
per_file = df_q.groupby(["source_type", "source_label"]).size().reset_index(name="n_questions")
per_file.sort_values(["source_type", "source_label"])

In [ ]:
# Inspect a specific question — change IDX
IDX = 0
q = questions[IDX]
print(f"q_id        : {q['q_id']}")
print(f"source_type : {q['source_type']}")
print(f"source_label: {q['source_label']}")
print(f"options     : {q['options']}")
print()
print("text:")
print(q["text"])

## 5. BM25 alignment — retrieval quality

In [ ]:
records = [json.loads(l) for l in (ALIGNED / "records.jsonl").open()]
df_rec = pd.DataFrame([
    {
        "question_id": r["question"]["id"],
        "source_type": r["question"]["metadata"].get("source_type"),
        "bm25_score": r["context"]["metadata"]["bm25_score"],
        "low_confidence": r["context"]["metadata"]["low_confidence"],
        "chunk_id": r["context"]["metadata"]["chunk_id"],
        "topic": r["context"]["metadata"]["topic"],
    }
    for r in records
])
print(f"Records: {len(df_rec)}")
print(f"Low-confidence alignments: {df_rec['low_confidence'].sum()}")
df_rec.describe()

In [ ]:
# BM25 score distribution
fig, ax = plt.subplots(figsize=(9, 4))
for src_type, grp in df_rec.groupby("source_type"):
    ax.hist(grp["bm25_score"], bins=30, alpha=0.6, label=src_type)
ax.axvline(5.0, color="red", linestyle="--", label="low-confidence threshold (5.0)")
ax.set_xlabel("BM25 score")
ax.set_ylabel("Count")
ax.set_title(f"BM25 alignment scores — {COURSE_ID}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Inspect a specific alignment — change IDX
IDX = 0
r = records[IDX]
print("=== Question ===")
print(r["question"]["text"])
print()
print(f"=== Retrieved chunk  (BM25={r['context']['metadata']['bm25_score']}) ===")
print(f"chunk_id: {r['context']['metadata']['chunk_id']}")
print(f"topic   : {r['context']['metadata']['topic']}")
print()
print(r["context"]["course_content"][:600])

## 6. Dataset summary

In [ ]:
dataset_rows = [json.loads(l) for l in DATASET.open() if not json.loads(l).get("_type")]

print("=== Dataset header ===")
header = json.loads(DATASET.open().readline())
print(json.dumps(header, indent=2))
print()
print(f"Total records      : {len(dataset_rows)}")
print(f"Courses            : {len(set(r['question']['metadata']['course_id'] for r in dataset_rows))}")

by_type = pd.Series(
    [r["question"]["metadata"].get("source_type") for r in dataset_rows]
).value_counts()
print()
print("By source type:")
print(by_type.to_string())